# Experiment: Phase B Head Program

Mục tiêu:
- Chạy đúng 4 line của `Pha B`: `B1`, `B2`, `B3`, `B4`.
- Tái dùng engine train của `R1` để giữ pipeline ổn định, resume-safe, cache-safe và tối ưu cho Colab T4.
- Chỉ thay những trục cần thiết theo spec: `head`, `backbone 16/256 hoặc 20/256`, và `sampling`.

Run matrix:
- `B1`: `16/256 + SimplifiedGlobalHead + band_balanced`
- `B2`: `16/256 + SimplifiedGlobalHead + sign_stratified`
- `B3`: `16/256 + RegimeSeparatedHead + best sampling from B1/B2`
- `B4`: `20/256 + winning head recipe from B1-B3`

Biến môi trường chính:
- `CHESS_PHASE_B_EXPERIMENT=B1|B2|B3|B4`
- `CHESS_PHASE_B_SOURCE_EXPERIMENT=B1|B2|B3` cho `B3/B4` khi cần
- `CHESS_PHASE_B_SOURCE_SAMPLING=band_balanced|sign_stratified` khi `B3` hoặc `B4(source=B3)` cần sampling cụ thể
- `CHESS_RUN_SUFFIX=<tag>` để tạo run_name mới
- `CHESS_EPOCHS_OVERRIDE=<int>` nếu cần run ngắn hơn để smoke-test
- `CHESS_STAGE_DATA_LOCAL=1` trên Colab để copy data từ Drive sang disk local


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    print('Mounted Google Drive.')
else:
    print('Running outside Colab.')


In [ ]:
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
import torch


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, '').strip().lower()
    if not raw:
        return bool(default)
    return raw in {'1', 'true', 'yes', 'y'}


def env_int(name: str, default=None):
    raw = os.environ.get(name, '').strip()
    if not raw:
        return default
    return int(raw)


def _candidate_repo_roots() -> list[Path]:
    candidates = []
    env_root = os.environ.get('CHESS_REPO_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    if IN_COLAB:
        candidates.append(Path('/content/drive/MyDrive/chess_engine'))
        shortcut_root = Path('/content/drive/.shortcut-targets-by-id')
        if shortcut_root.exists():
            for child in shortcut_root.iterdir():
                candidates.append(child / 'chess_engine')
    seen = set()
    out = []
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        out.append(path)
    return out


def resolve_repo_root() -> Path:
    for candidate in _candidate_repo_roots():
        helper_path = candidate / 'train_v5_phaseB' / 'phase_b_train_helpers.py'
        if helper_path.exists():
            return candidate.resolve()
    raise RuntimeError('Cannot resolve repository root. Set CHESS_REPO_ROOT explicitly.')


REPO_ROOT = resolve_repo_root()
RUNS_ROOT = Path(os.environ.get('CHESS_RUNS_ROOT', '').strip()) if os.environ.get('CHESS_RUNS_ROOT', '').strip() else (REPO_ROOT / 'runs')
HELPER_PATH = REPO_ROOT / 'train_v5_phaseB' / 'phase_b_train_helpers.py'
PHASE_B_EXPERIMENT = os.environ.get('CHESS_PHASE_B_EXPERIMENT', 'B1').strip().upper()
PHASE_B_SOURCE_EXPERIMENT = os.environ.get('CHESS_PHASE_B_SOURCE_EXPERIMENT', '').strip().upper() or None
PHASE_B_SOURCE_SAMPLING = os.environ.get('CHESS_PHASE_B_SOURCE_SAMPLING', '').strip().lower() or None
RUN_SUFFIX = os.environ.get('CHESS_RUN_SUFFIX', '').strip()
EPOCHS_OVERRIDE = env_int('CHESS_EPOCHS_OVERRIDE', None)
AUTOTUNE_PROFILE = env_bool('CHESS_AUTOTUNE_PROFILE', True)
DATA_ROOT_OVERRIDE = os.environ.get('CHESS_DATA_ROOT', '').strip()
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else (REPO_ROOT / 'data' / 'process')
STAGE_DATA_LOCAL = IN_COLAB and env_bool('CHESS_STAGE_DATA_LOCAL', True)
FORCE_RESTAGE = IN_COLAB and env_bool('CHESS_FORCE_RESTAGE', False)

if STAGE_DATA_LOCAL:
    DATA_ROOT_ACTIVE = Path('/content/chess_engine_data/process')
    if FORCE_RESTAGE and DATA_ROOT_ACTIVE.exists():
        shutil.rmtree(DATA_ROOT_ACTIVE)
    if not DATA_ROOT_ACTIVE.exists():
        DATA_ROOT_ACTIVE.parent.mkdir(parents=True, exist_ok=True)
        print(f'Staging data from {DATA_ROOT} -> {DATA_ROOT_ACTIVE}')
        shutil.copytree(DATA_ROOT, DATA_ROOT_ACTIVE)
    else:
        print(f'Reusing staged data at {DATA_ROOT_ACTIVE}')
else:
    DATA_ROOT_ACTIVE = DATA_ROOT

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = None
GPU_CAPABILITY = None
GPU_TOTAL_MEM_GB = None
TF32_SUPPORTED = False
NVIDIA_SMI_SUMMARY = 'n/a'
CPU_COUNT = os.cpu_count() or 1

if torch.cuda.is_available():
    gpu_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(gpu_index)
    GPU_NAME = props.name
    GPU_CAPABILITY = torch.cuda.get_device_capability(gpu_index)
    GPU_TOTAL_MEM_GB = props.total_memory / (1024 ** 3)
    TF32_SUPPORTED = bool(GPU_CAPABILITY[0] >= 8)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = TF32_SUPPORTED
    torch.backends.cudnn.allow_tf32 = TF32_SUPPORTED

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        check=True,
        capture_output=True,
        text=True,
    )
    NVIDIA_SMI_SUMMARY = result.stdout.strip().splitlines()[0].strip()
except Exception:
    pass

print('REPO_ROOT =', REPO_ROOT)
print('RUNS_ROOT =', RUNS_ROOT)
print('DATA_ROOT_ACTIVE =', DATA_ROOT_ACTIVE)
print('DEVICE =', DEVICE)
print('GPU_CAPABILITY =', GPU_CAPABILITY)
print('TF32_SUPPORTED =', TF32_SUPPORTED)
print('GPU_NAME =', GPU_NAME)
print('GPU_TOTAL_MEM_GB =', GPU_TOTAL_MEM_GB)
print('NVIDIA_SMI_SUMMARY =', NVIDIA_SMI_SUMMARY)
print('CPU_COUNT =', CPU_COUNT)

if DEVICE != 'cuda':
    raise RuntimeError('CUDA is required for Phase B training.')


In [ ]:
def import_phase_b_helper(helper_path: Path):
    spec = importlib.util.spec_from_file_location('train_v5_phaseB_phase_b_train_helpers', helper_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot import helper from {helper_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


lab = import_phase_b_helper(HELPER_PATH)
experiment = lab.build_phase_b_experiment(
    PHASE_B_EXPERIMENT,
    source_experiment=PHASE_B_SOURCE_EXPERIMENT,
    source_sampling_mode=PHASE_B_SOURCE_SAMPLING,
    run_suffix=RUN_SUFFIX,
    epochs_override=EPOCHS_OVERRIDE,
)
exp_info = lab.describe_phase_b_experiment(experiment)
print(json.dumps(exp_info, indent=2, ensure_ascii=False))

RUN_DIR = RUNS_ROOT / experiment.train_cfg.run_name
existing_config_path = RUN_DIR / 'reports' / 'run_config.json'
if existing_config_path.exists():
    existing = json.loads(existing_config_path.read_text(encoding='utf-8'))
    existing_runtime = existing.get('runtime', {}) if isinstance(existing, dict) else {}
    existing_gpu = str(existing_runtime.get('gpu_name', ''))
    if existing_gpu and GPU_NAME and existing_gpu != GPU_NAME:
        print(f'[warning] Existing run_name was created on GPU={existing_gpu}, current GPU={GPU_NAME}.')


In [ ]:
run_artifacts = lab.run_phase_b_training(
    repo_root=REPO_ROOT,
    runs_root=RUNS_ROOT,
    data_root=DATA_ROOT_ACTIVE,
    experiment_id=PHASE_B_EXPERIMENT,
    source_experiment=PHASE_B_SOURCE_EXPERIMENT,
    source_sampling_mode=PHASE_B_SOURCE_SAMPLING,
    autotune_profile=AUTOTUNE_PROFILE,
    run_suffix=RUN_SUFFIX,
    epochs_override=EPOCHS_OVERRIDE,
)

print('Selected checkpoint:', run_artifacts['selected_checkpoint'])
print('Reports dir:', run_artifacts['paths']['reports_dir'])


In [ ]:
reports_dir = RUN_DIR / 'reports'
history_df = pd.read_csv(reports_dir / 'history.csv')
step_history_path = reports_dir / 'step_history.csv'
step_hist = pd.read_csv(step_history_path) if step_history_path.exists() else pd.DataFrame()
decision = json.loads((reports_dir / 'decision_summary.json').read_text(encoding='utf-8'))
l4_ref = json.loads((reports_dir / 'l4_reference.json').read_text(encoding='utf-8'))
selected_eval = json.loads((reports_dir / 'selected_checkpoint_eval.json').read_text(encoding='utf-8')) if (reports_dir / 'selected_checkpoint_eval.json').exists() else None
autotune_path = reports_dir / 'train_batch_autotune.json'
autotune = json.loads(autotune_path.read_text(encoding='utf-8')) if autotune_path.exists() else None

mid_gate = l4_ref['primary']['oracle_midband_mae_sum_stable'] * 1.05
slope_gate = l4_ref['primary']['oracle_stable_0.7_slope'] - 0.02
l4_center = l4_ref['primary']['center_score']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(history_df['epoch'], history_df['train_total_objective'], label='train_total_objective')
axes[0, 0].plot(history_df['epoch'], history_df['train_main_objective'], label='train_main_objective')
axes[0, 0].plot(history_df['epoch'], history_df['train_aux_objective'], label='train_aux_objective')
axes[0, 0].set_title(f'{PHASE_B_EXPERIMENT} Train Objectives')
axes[0, 0].set_xlabel('epoch')
axes[0, 0].set_ylabel('objective')
axes[0, 0].legend()

ax2 = axes[0, 1]
ax2.plot(history_df['epoch'], history_df['oracle_midband_mae_sum_stable'], marker='o', label=f'{PHASE_B_EXPERIMENT} midband MAE')
ax2.axhline(mid_gate, color='tab:olive', linestyle='--', label='L4 midband gate')
ax2.set_xlabel('epoch')
ax2.set_ylabel('oracle_midband_mae_sum_stable')
ax2.set_title('Hard A-Gate Metrics')
ax2b = ax2.twinx()
ax2b.plot(history_df['epoch'], history_df['oracle_stable_0.7_slope'], marker='s', color='tab:green', label=f'{PHASE_B_EXPERIMENT} slope')
ax2b.axhline(slope_gate, color='goldenrod', linestyle='--', label='L4 slope gate')
ax2b.set_ylabel('oracle_stable_0.7_slope')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

axes[1, 0].plot(history_df['epoch'], history_df['center_score'], marker='o', label=f'{PHASE_B_EXPERIMENT} center_score')
axes[1, 0].axhline(l4_center, color='gray', linestyle='--', label='L4 center_score')
axes[1, 0].set_title('Center Score vs L4')
axes[1, 0].set_xlabel('epoch')
axes[1, 0].set_ylabel('center_score')
axes[1, 0].legend()

ax4 = axes[1, 1]
if not step_hist.empty:
    ax4.plot(step_hist['global_step'], step_hist['grad_cosine_backbone'], label='cosine_pre')
    ax4.plot(step_hist['global_step'], step_hist['grad_cosine_backbone_post'], label='cosine_post')
    ax4.set_xlabel('global_step')
    ax4.set_ylabel('cosine')
    ax4.set_title('Backbone Gradient Geometry')
    ax4.legend()
else:
    ax4.text(0.5, 0.5, 'No step_history.csv', ha='center', va='center')
    ax4.set_axis_off()

plt.tight_layout()
plt.show()

summary = {
    'experiment': exp_info['experiment_id'],
    'run_name': experiment.train_cfg.run_name,
    'selected_checkpoint_if_stop_now': decision.get('selected_checkpoint_if_stopped_now'),
    'has_best_gate': decision.get('has_best_gate'),
    'best_any_center_score': decision.get('best_any_center_score'),
    'best_pareto_A_midband': decision.get('best_pareto_A_midband'),
    'best_pareto_A_slope': decision.get('best_pareto_A_slope'),
    'last_epoch': decision.get('last_epoch'),
    'last_epoch_tags': decision.get('last_epoch_tags'),
    'selected_eval_center_score': None if selected_eval is None else selected_eval.get('center_score'),
    'selected_eval_midband': None if selected_eval is None else selected_eval.get('oracle_midband_mae_sum_stable'),
    'selected_eval_slope': None if selected_eval is None else selected_eval.get('oracle_stable_0.7_slope'),
    'autotune_selected_profile': None if autotune is None else autotune.get('selected_profile'),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
best_mid_idx = int(history_df['oracle_midband_mae_sum_stable'].idxmin())
best_slope_idx = int(history_df['oracle_stable_0.7_slope'].idxmax())
best_center_idx = int(history_df['center_score'].idxmin())

print('Best midband epoch:', int(history_df.loc[best_mid_idx, 'epoch']), float(history_df.loc[best_mid_idx, 'oracle_midband_mae_sum_stable']))
print('Best slope epoch:', int(history_df.loc[best_slope_idx, 'epoch']), float(history_df.loc[best_slope_idx, 'oracle_stable_0.7_slope']))
print('Best center epoch:', int(history_df.loc[best_center_idx, 'epoch']), float(history_df.loc[best_center_idx, 'center_score']))

if len(history_df) >= 3:
    print('\nLast 3 epochs:')
    display(history_df.tail(3)[['epoch', 'lr', 'oracle_midband_mae_sum_stable', 'oracle_stable_0.7_slope', 'center_score', 'train_total_objective']])
